In [0]:
# Creating a database to hold all our tables

spark.sql("CREATE DATABASE IF NOT EXISTS diabetes_medallion")
spark.sql("USE diabetes_medallion")

print("✅ Database created and ready!")

In [0]:
SOURCE_PATH     = "/Volumes/workspace/diabetes_pipeline/healthcare_db/"
BRONZE_TABLE    = "diabetes_medallion.bronze_diabetes"
CHECKPOINT_PATH = "/Volumes/workspace/diabetes_pipeline/healthcare_db/_checkpoints/bronze"

print(f"✅ Source path:     {SOURCE_PATH}")
print(f"✅ Bronze table:    {BRONZE_TABLE}")
print(f"✅ Checkpoint path: {CHECKPOINT_PATH}")

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col
import uuid

# Generate unique batch ID for this run
batch_id = str(uuid.uuid4())

# READ using Autoloader
raw_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "/schema") \
    .load(SOURCE_PATH)

# Add audit columns
# NOTE: Unity Catalog requires _metadata.file_path 
#       instead of input_file_name()
bronze_stream = raw_stream \
    .withColumn("ingestion_timestamp", current_timestamp())          \
    .withColumn("source_file",         col("_metadata.file_path"))   \
    .withColumn("batch_id",            lit(batch_id))

# WRITE to Bronze Delta table
query = bronze_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable(BRONZE_TABLE)

query.awaitTermination()
print("✅ Autoloader finished processing!")
print(f"   Batch ID: {batch_id}")

In [0]:
# Read back from Bronze table to confirm data was written
bronze_df = spark.table(BRONZE_TABLE)

print("=" * 50)
print("      BRONZE TABLE VERIFICATION")
print("=" * 50)
print(f"✅ Total Rows:    {bronze_df.count():,}")
print(f"✅ Total Columns: {len(bronze_df.columns)}")

# Confirm which file was processed
print("\n=== FILE PROCESSED ===")
bronze_df.select("source_file") \
         .distinct() \
         .show(truncate=False)

# Confirm audit columns exist
print("=== SAMPLE WITH AUDIT COLUMNS ===")
bronze_df.select(
    "Diabetes_binary", 
    "BMI", 
    "Age",
    "ingestion_timestamp", 
    "batch_id",
    "source_file"
).show(5, truncate=False)

In [0]:
# ================================================
# Delta Lake Implementation
# This notebook demonstrates the following points for:
# ✅ ACID transactions
# ✅ Table history / versioning
# ✅ OPTIMIZE for performance
# ================================================

# 1. Show table history (versioning / time travel)
print("=== BRONZE TABLE HISTORY ===")
spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}").show(truncate=False)

# 2. Optimize the table for faster reads downstream
print("=== OPTIMIZING BRONZE TABLE ===")
spark.sql(f"OPTIMIZE {BRONZE_TABLE}")
print("✅ OPTIMIZE complete!")

# 3. Show table details
print("=== TABLE DETAILS ===")
spark.sql(f"DESCRIBE DETAIL {BRONZE_TABLE}").show(truncate=False)


In [0]:
# Let's see exactly what columns exist in your Bronze table
bronze_df = spark.table("diabetes_medallion.bronze_diabetes")

print("=== ALL COLUMNS IN BRONZE TABLE ===")
for i, col in enumerate(bronze_df.columns, 1):
    print(f"  {i}. {col}")

print(f"\nTotal columns: {len(bronze_df.columns)}")

In [0]:
# Check if _rescued_data has any values
# If all NULL = all data loaded perfectly

from pyspark.sql.functions import col, count, sum as spark_sum

bronze_df = spark.table("diabetes_medallion.bronze_diabetes")

rescued = bronze_df.filter(
    col("_rescued_data").isNotNull()
).count()

print(f"Rows with rescued data: {rescued}")

if rescued == 0:
    print("✅ PERFECT — All data loaded cleanly!")
    print("   _rescued_data is NULL for all rows")
else:
    print(f"⚠️  {rescued} rows had schema mismatches")
    print("   Let's investigate those rows")